# "The Salary Is Right" — QLoRA Fine-Tuning

## Week 7 Community Contribution: Omotosho Joseph

Fine-tune Llama 3.2 3B with QLoRA to predict annual salary from job postings.

**Dataset:** `lukebarousse/data_jobs` — 786K real-world job postings from 2023

**Pipeline:**
1. **Data Preparation** — Load, parse, deduplicate, format as prompt/completion, push to HuggingFace
2. **Base Model Evaluation** — See how un-fine-tuned Llama 3.2 3B performs
3. **QLoRA Training** — Fine-tune with LoRA adapters using SFTTrainer
4. **Fine-Tuned Evaluation** — Load adapters and evaluate on the test set

---

### Instructions

This notebook has **4 parts**. You can run them in order, but between
Part 3 (Training) and Part 4 (Evaluation) you should **restart the runtime**
to free GPU memory.

- **LITE_MODE=True**: Free T4, 1 epoch, attention-only LoRA (r=32)
- **LITE_MODE=False**: Paid A100 (high RAM), 3 epochs, full LoRA (r=256)

In [ ]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1 datasets pydantic transformers huggingface_hub wandb plotly scikit-learn

In [ ]:
import os
import re
import math
import random
from tqdm.auto import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from datasets import load_dataset, Dataset, DatasetDict
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_squared_error, r2_score
from itertools import accumulate
from IPython.display import clear_output
from pydantic import BaseModel
from typing import Optional, Self
import wandb

---
## Global Constants

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "salary"
HF_USER = "YOUR_HF_USERNAME"  # <-- CHANGE THIS to your HuggingFace username

LITE_MODE = True

QUANT_4_BIT = True
MAX_PROMPT_TOKENS = 200

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

---
## Helper Classes

The `Job` model and parser (adapted from Week 6), plus the `Tester`
evaluator (adapted for Colab dict-based data).

In [ ]:
# ── Job model & parser ──────────────────────────────────────────────

PREFIX = "Salary is $"
QUESTION = "What is the annual salary for this position to the nearest thousand dollars?"
MIN_CHARS = 100
MIN_SALARY = 15_000
MAX_SALARY = 500_000
MAX_TEXT = 3000


class Job(BaseModel):
    title: str
    category: str
    salary: float
    full: Optional[str] = None
    location: Optional[str] = None
    company: Optional[str] = None
    skills: Optional[str] = None
    schedule_type: Optional[str] = None
    work_from_home: Optional[bool] = None
    summary: Optional[str] = None
    prompt: Optional[str] = None
    completion: Optional[str] = None
    id: Optional[int] = None

    def __repr__(self) -> str:
        return f"<{self.title} = ${self.salary:,.0f}>"

    def make_prompts(self, tokenizer, max_tokens, do_round):
        text = self.summary or self.full or ""
        tokens = tokenizer.encode(text, add_special_tokens=False)
        if len(tokens) > max_tokens:
            text = tokenizer.decode(tokens[:max_tokens]).rstrip()
        self.prompt = f"{QUESTION}\n\n{text}\n\n{PREFIX}"
        if do_round:
            self.completion = f"{round(self.salary / 1000) * 1000:.0f}"
        else:
            self.completion = f"{self.salary:.0f}"

    def count_prompt_tokens(self, tokenizer):
        full = self.prompt + self.completion
        return len(tokenizer.encode(full, add_special_tokens=False))

    def to_datapoint(self) -> dict:
        return {"prompt": self.prompt, "completion": self.completion}

    @staticmethod
    def push_prompts_to_hub(dataset_name, train, val, test):
        DatasetDict({
            "train": Dataset.from_list([j.to_datapoint() for j in train]),
            "val":   Dataset.from_list([j.to_datapoint() for j in val]),
            "test":  Dataset.from_list([j.to_datapoint() for j in test]),
        }).push_to_hub(dataset_name)


def _simplify(text):
    if text is None:
        return ""
    return str(text).replace("\n", " ").replace("\r", "").replace("\t", " ").replace("  ", " ").strip()[:MAX_TEXT]


def _scrub(title, company, location, skills, schedule_type):
    result = f"Job Title: {title}\n"
    if company:       result += f"Company: {_simplify(company)}\n"
    if location:      result += f"Location: {_simplify(location)}\n"
    if schedule_type: result += f"Type: {_simplify(schedule_type)}\n"
    if skills:        result += f"Skills: {_simplify(skills)}\n"
    return result.strip()[:MAX_TEXT]


def parse(row):
    try:
        salary = row.get("salary_year_avg")
        if salary is None or salary != salary:
            return None
        salary = float(salary)
    except (ValueError, TypeError):
        return None
    if not (MIN_SALARY <= salary <= MAX_SALARY):
        return None
    title = row.get("job_title") or row.get("job_title_short") or ""
    if not title:
        return None
    company = row.get("company_name") or ""
    location = row.get("job_location") or ""
    category = row.get("job_title_short") or "Other"
    skills = row.get("job_skills") or ""
    schedule_type = row.get("job_schedule_type") or ""
    work_from_home = bool(row.get("job_work_from_home", False))
    full = _scrub(title, company, location, skills, schedule_type)
    if len(full) < MIN_CHARS:
        return None
    return Job(
        title=title, category=category, salary=salary, full=full,
        location=location, company=company, skills=skills,
        schedule_type=schedule_type, work_from_home=work_from_home,
    )

In [ ]:
# ── Evaluator (Colab-compatible, dict-based data) ─────────────────

GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"
COLOR_MAP = {"red": RED, "orange": YELLOW, "green": GREEN}
DEFAULT_SIZE = 200


class Tester:
    def __init__(self, predictor, data, title=None, size=DEFAULT_SIZE):
        self.predictor = predictor
        self.data = data
        self.title = title or predictor.__name__.replace("__", ".").replace("_", " ").title()
        self.size = size
        self.titles = []
        self.guesses = []
        self.truths = []
        self.errors = []
        self.colors = []

    @staticmethod
    def post_process(value):
        if isinstance(value, str):
            value = value.replace("$", "").replace(",", "").replace("k", "000").replace("K", "000")
            match = re.search(r"[-+]?\d*\.?\d+", value)
            return float(match.group()) if match else 0
        return value

    def color_for(self, error, truth):
        if error < 15000 or error / truth < 0.15:
            return "green"
        elif error < 40000 or error / truth < 0.3:
            return "orange"
        return "red"

    def run_datapoint(self, i):
        dp = self.data[i]
        value = self.predictor(dp)
        guess = self.post_process(value)
        truth = float(dp["completion"])
        error = abs(guess - truth)
        color = self.color_for(error, truth)
        pieces = dp["prompt"].split("Job Title: ")
        title = pieces[1].split("\n")[0] if len(pieces) > 1 else pieces[0][:40]
        title = title if len(title) <= 40 else title[:40] + "..."
        return title, guess, truth, error, color

    def chart(self, title):
        df = pd.DataFrame({
            "truth": self.truths, "guess": self.guesses,
            "title": self.titles, "error": self.errors, "color": self.colors,
        })
        df["hover"] = [
            f"{t}\nGuess=${g:,.0f} Actual=${y:,.0f}"
            for t, g, y in zip(df["title"], df["guess"], df["truth"])
        ]
        max_val = float(max(df["truth"].max(), df["guess"].max()))
        fig = px.scatter(
            df, x="truth", y="guess", color="color",
            color_discrete_map={"green": "green", "orange": "orange", "red": "red"},
            title=title,
            labels={"truth": "Actual Salary ($)", "guess": "Predicted Salary ($)"},
            width=800, height=600,
        )
        for tr in fig.data:
            mask = df["color"] == tr.name
            tr.customdata = df.loc[mask, ["hover"]].to_numpy()
            tr.hovertemplate = "%{customdata[0]}<extra></extra>"
            tr.marker.update(size=6)
        fig.add_trace(go.Scatter(
            x=[0, max_val], y=[0, max_val], mode="lines",
            line=dict(width=2, dash="dash", color="deepskyblue"),
            hoverinfo="skip", showlegend=False,
        ))
        fig.update_xaxes(range=[0, max_val])
        fig.update_yaxes(range=[0, max_val])
        fig.update_layout(showlegend=False)
        fig.show()

    def error_trend_chart(self):
        n = len(self.errors)
        running_sums = list(accumulate(self.errors))
        x = list(range(1, n + 1))
        running_means = [s / i for s, i in zip(running_sums, x)]
        running_squares = list(accumulate(e * e for e in self.errors))
        running_stds = [
            math.sqrt((sq / i) - (m ** 2)) if i > 1 else 0
            for i, sq, m in zip(x, running_squares, running_means)
        ]
        ci = [1.96 * (sd / math.sqrt(i)) if i > 1 else 0 for i, sd in zip(x, running_stds)]
        upper = [m + c for m, c in zip(running_means, ci)]
        lower = [m - c for m, c in zip(running_means, ci)]
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=x + x[::-1], y=upper + lower[::-1], fill="toself",
            fillcolor="rgba(128,128,128,0.2)",
            line=dict(color="rgba(255,255,255,0)"),
            hoverinfo="skip", showlegend=False,
        ))
        fig.add_trace(go.Scatter(
            x=x, y=running_means, mode="lines",
            line=dict(width=3, color="firebrick"),
            name="Cumulative Avg Error",
            customdata=list(zip(ci,)),
            hovertemplate="n=%{x}<br>Avg Error=$%{y:,.0f}<br>±95%25 CI=$%{customdata[0]:,.0f}<extra></extra>",
        ))
        fig.update_layout(
            title=f"{self.title} Error: ${running_means[-1]:,.0f} ± ${ci[-1]:,.0f}",
            xaxis_title="Datapoints", yaxis_title="Avg Absolute Error ($)",
            width=800, height=300, template="plotly_white", showlegend=False,
        )
        fig.show()

    def report(self):
        avg_err = sum(self.errors) / self.size
        mse = mean_squared_error(self.truths, self.guesses)
        r2 = r2_score(self.truths, self.guesses) * 100
        title = f"{self.title}<br><b>Error:</b> ${avg_err:,.0f} <b>MSE:</b> {mse:,.0f} <b>r²:</b> {r2:.1f}%"
        self.error_trend_chart()
        self.chart(title)

    def run(self):
        for i in tqdm(range(self.size)):
            title, guess, truth, error, color = self.run_datapoint(i)
            self.titles.append(title)
            self.guesses.append(guess)
            self.truths.append(truth)
            self.errors.append(error)
            self.colors.append(color)
            print(f"{COLOR_MAP[color]}${error:,.0f} ", end="")
        clear_output(wait=True)
        self.report()


def evaluate(function, data, size=DEFAULT_SIZE):
    Tester(function, data, size=size).run()

---
# Part 1: Data Preparation

Load `lukebarousse/data_jobs`, parse into Job objects, build prompt/completion
pairs, and push to HuggingFace Hub.

In [ ]:
raw_dataset = load_dataset('lukebarousse/data_jobs', split='train')
print(f"Total job postings: {len(raw_dataset):,}")

items = []
for i in tqdm(range(len(raw_dataset))):
    job = parse(raw_dataset[i])
    if job is not None:
        items.append(job)

print(f"Parsed {len(items):,} valid jobs")

In [ ]:
# Deduplicate
seen = set()
unique_items = []
for item in items:
    key = (item.title, item.company, item.salary)
    if key not in seen:
        seen.add(key)
        unique_items.append(item)
items = unique_items
print(f"After dedup: {len(items):,}")

# Use structured text as summary
for item in items:
    if not item.summary:
        item.summary = item.full

# Shuffle and split
random.seed(42)
random.shuffle(items)

n = len(items)
val_size = min(2000, int(n * 0.05))
test_size = min(2000, int(n * 0.05))
train_size = n - val_size - test_size

train_jobs = items[:train_size]
val_jobs   = items[train_size:train_size + val_size]
test_jobs  = items[train_size + val_size:]

print(f"Train: {len(train_jobs):,} | Val: {len(val_jobs):,} | Test: {len(test_jobs):,}")

In [ ]:
# Build prompt/completions using tokenizer for truncation
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

for job in train_jobs:
    job.make_prompts(tokenizer, MAX_PROMPT_TOKENS, do_round=True)
for job in val_jobs:
    job.make_prompts(tokenizer, MAX_PROMPT_TOKENS, do_round=True)
for job in test_jobs:
    job.make_prompts(tokenizer, MAX_PROMPT_TOKENS, do_round=False)

print("=== Train example ===")
print(f"Prompt: {train_jobs[0].prompt}")
print(f"Completion: {train_jobs[0].completion}")
print(f"\n=== Test example ===")
print(f"Prompt: {test_jobs[0].prompt}")
print(f"Completion: {test_jobs[0].completion}")

In [ ]:
# Token length distribution
token_counts = [job.count_prompt_tokens(tokenizer) for job in train_jobs]
plt.hist(token_counts, bins=50, color='skyblue', edgecolor='black')
plt.title(f"Token counts — max={max(token_counts)}, mean={sum(token_counts)/len(token_counts):.0f}")
plt.xlabel("Tokens")
plt.ylabel("Count")
plt.show()

In [ ]:
# Push to HuggingFace Hub
PROMPT_DATASET = f"{HF_USER}/salary_prompts"

Job.push_prompts_to_hub(PROMPT_DATASET, train_jobs, val_jobs, test_jobs)
print(f"\nPushed to https://huggingface.co/datasets/{PROMPT_DATASET}")

---
# Part 2: Base Model Evaluation

Load the 4-bit quantized Llama 3.2 3B base model and evaluate it
on our test set — **before** any fine-tuning.

This is our baseline. Expect poor results since the base model has
no special knowledge of salary prediction.

In [ ]:
DATASET_NAME = f"{HF_USER}/salary_prompts"
dataset = load_dataset(DATASET_NAME)
test_data = dataset['test']

print(f"Test set: {len(test_data)} examples")
print(test_data[0])

In [ ]:
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
        bnb_4bit_quant_type="nf4"
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    )

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
def base_model_predict(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = base_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

# Quick test
result = base_model_predict(test_data[0])
print(f"Prediction: {result}")
print(f"Actual: {test_data[0]['completion']}")

In [ ]:
set_seed(42)
evaluate(base_model_predict, test_data)

---
# Part 3: QLoRA Training

⚠️ **Before running this section**, restart the runtime to free GPU memory
from the base model evaluation above.

**Runtime → Restart session**, then re-run from the top *up to but not including* Part 2.
Alternatively, just run the imports and constants cells, then jump here.

In [ ]:
DATASET_NAME = f"{HF_USER}/salary_prompts"

RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
if LITE_MODE:
    RUN_NAME += "-lite"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# ── Hyperparameters ─────────────────────────────────────────────

EPOCHS = 1 if LITE_MODE else 3
BATCH_SIZE = 32 if LITE_MODE else 256
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1

LORA_R = 32 if LITE_MODE else 256
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS if LITE_MODE else ATTENTION_LAYERS + MLP_LAYERS
LORA_DROPOUT = 0.1

LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

VAL_SIZE = 500 if LITE_MODE else 1000
LOG_STEPS = 5 if LITE_MODE else 10
SAVE_STEPS = 100 if LITE_MODE else 200
LOG_TO_WANDB = True

print(f"Mode: {'LITE' if LITE_MODE else 'FULL'} | BF16: {use_bf16}")
print(f"Run: {HUB_MODEL_NAME}")
print(f"LoRA: r={LORA_R}, alpha={LORA_ALPHA}, targets={TARGET_MODULES}")

In [ ]:
# Weights & Biases login
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
val = dataset['val'].select(range(VAL_SIZE))
test = dataset['test']

print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
print(f"\nSample: {train[0]['prompt'][:120]}...")
print(f"Completion: {train[0]['completion']}")

In [ ]:
if LOG_TO_WANDB:
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)

In [ ]:
if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
        bnb_4bit_quant_type="nf4"
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    )

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
)

In [ ]:
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters,
)

In [ ]:
# 🚀 Train!
fine_tuning.train()

fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"\n✅ Saved to the hub: {HUB_MODEL_NAME}")

In [ ]:
if LOG_TO_WANDB:
    wandb.finish()

print(f"\nIMPORTANT — Copy these values for Part 4:")
print(f"  RUN_NAME = \"{RUN_NAME}\"")
print(f"  HUB_MODEL_NAME = \"{HUB_MODEL_NAME}\"")
print(f"\nFor FULL mode, go to https://huggingface.co/{HUB_MODEL_NAME}/commits/main")
print(f"and find the commit hash for the checkpoint with the lowest eval_loss.")

---
# Part 4: Evaluate the Fine-Tuned Model

⚠️ **Restart the runtime** before running this section to free GPU memory.

Then re-run **only** the pip install, imports, constants, and helper cells
at the top, then skip straight to this section.

Fill in `RUN_NAME` and `REVISION` from your training run.

In [ ]:
DATASET_NAME = f"{HF_USER}/salary_prompts"

# ──────────────────────────────────────────────────────────────────
# PASTE YOUR VALUES HERE from the training output:
# ──────────────────────────────────────────────────────────────────

if LITE_MODE:
    RUN_NAME = "YYYY-MM-DD_HH.MM.SS-lite"   # <-- paste your lite run name
    REVISION = None
else:
    RUN_NAME = "YYYY-MM-DD_HH.MM.SS"        # <-- paste your full run name
    REVISION = "your_commit_hash_here"       # <-- best checkpoint commit hash

# ──────────────────────────────────────────────────────────────────

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

print(f"Loading: {HUB_MODEL_NAME}")
if REVISION:
    print(f"Revision: {REVISION}")

In [ ]:
dataset = load_dataset(DATASET_NAME)
test = dataset['test']
print(f"Test set: {len(test)} examples")
print(test[0])

In [ ]:
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
        bnb_4bit_quant_type="nf4"
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    )

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# Load LoRA adapters
if REVISION:
    fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
else:
    fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)

print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")

In [ ]:
# Inspect the adapted model
fine_tuned_model

In [ ]:
def model_predict(item):
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)

# Quick sanity check
result = model_predict(test[0])
print(f"Prediction: {result}")
print(f"Actual: {test[0]['completion']}")

In [ ]:
set_seed(42)
evaluate(model_predict, test)

---
## Next Steps

1. Compare the fine-tuned error against the base model error and your Week 6 frontier LLM results
2. Try `LITE_MODE=False` on an A100 for better performance
3. Experiment with hyperparameters: `LORA_R`, `LEARNING_RATE`, `EPOCHS`, `BATCH_SIZE`
4. Use W&B to find the best checkpoint (lowest eval_loss) and set `REVISION`
5. Try manipulating the data — different prompt formats, more/less text, etc.